# Forecasting experiments
Predict recorded monthly prescription-item activity at horizons 1–3. Fixed England atorvastatin data, 2023–2025; this does not predict patients, revenue or India demand.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image
root=Path.cwd()
if root.name=='notebooks': root=root.parent
sys.path.insert(0,str(root))
from src.forecast_config import configuration
from src.forecast_models import fit_forecast
from src.forecast_features import supervised_pairs
from src.forecast_metrics import metrics
cfg=configuration(root)
def table(name): return pd.read_parquet(root/f'outputs/phase4/tables/{name}.parquet')
def chart(name): display(Image(filename=str(root/f'outputs/phase4/charts/{name}.png')))
national=pd.read_parquet(root/'outputs/tables/national_monthly_trends.parquet').set_index('month').prescription_items.sort_index()


## Chronological design and training-only eligibility
The development data end June 2025. Holdout outcomes July–December cannot select models. The initial 24 months freeze volume/coverage eligibility; the full-history Phase 2 inventory is descriptive. Holdout origins begin June 2025, so horizon samples are triangular (6/5/4), without pre-freeze origins.

In [ ]:
display(table('series_eligibility').query('eligible'))
display(table('development_folds'))
display(table('holdout_folds'))

## Baselines at one genuine origin
Naive uses the last value; seasonal naive uses the target month a year earlier; drift extends the first-to-last slope. All three future steps are issued from the same cutoff.

In [ ]:
train=national.loc[:'2024-12-01']
for model in ['naive','seasonal_naive','drift']:
    raw,point,parameters,_=fit_forecast(train,[1,2,3],{'model':model,'variant':model,'alpha':None},cfg)
    display({'model':model,'origin':str(train.index[-1]),'prediction':point.tolist(),'parameters':parameters})

## Damped ETS
Additive damped trend, no seasonality, estimated initialization from the training slice. Optimization failures remain unavailable and are logged; they are not relabeled fallback forecasts.

In [ ]:
raw,point,parameters,_=fit_forecast(train,[1,2,3],{'model':'damped_ets','variant':'damped_ets','alpha':None},cfg)
display(parameters)
display(point)
display(table('model_failures')[['series_id','origin_date','horizon','failure_reason']])

## Direct Ridge and leakage prevention
At feature origin t, lag_1 is y(t), lag_2=y(t−1), lag_3=y(t−2). Training labels y(t+h) must already be observed by the cutoff. Calendar sine/cosine are known ahead. StandardScaler fits only supervised training rows. Six compact features leave 21/20/19 rows at the first origin; alpha comes from development folds.

In [ ]:
x,y,provenance=supervised_pairs(train,3,cfg)
display(x.head())
display(provenance.tail())
assert provenance.target_date.max()<=train.index[-1]
_,point,parameters,_=fit_forecast(train,[1,2,3],{'model':'ridge','variant':'ridge_alpha_10','alpha':10},cfg)
display(parameters)
display(table('development_configuration_metrics'))

## Frozen model choice
Shared pooled-level MASE selection, with 2% simplicity tie tolerance. Any development failure excludes a configuration from selection for that level. Final holdout scores cannot change the choices.

In [ ]:
display(table('model_selection_decisions').drop_duplicates(['aggregation_level','selected_variant']))
chart('01_national_history_origins')
chart('12_development_holdout')